# Task 4 - Complete comparison on the Synthetic-NL model (Sec IV-C)

Single canonical, reproducible implementation. Five estimators x two scenarios
(full / partial information), reported as posterior state MSE (dB).

**Methods:** EKF, Robust EKF (REKF), KalmanNet, existing Robust KalmanNet
(professor's original), proposed Robust KalmanNet (GRU-learned `c`, BPTT).

**Professor's expected qualitative orderings** (the acceptance target):
- full: `EKF >= RobustEKF`, `KalmanNet >= RobustKalmanNet`
- partial: `RobustEKF > EKF`, `RobustKalmanNet > KalmanNet`

**Design decisions applied** (see `docs/TASK4_DESIGN.md`):
- aligned filter init `x0 = ones` (the generative init), off the `x=0`
  degenerate-linearization trap; common initial covariance `1e-3*I`;
- partial information = paper's `f`-only mismatch (alpha=beta=1, phi=delta=0);
  data always from the TRUE model; KalmanNet trained on the TRUE data;
- Robust EKF reported at its best `c` from a sweep; proposed net learns `c_t`.

Run top-to-bottom. Trainings (KalmanNet x2, proposed x2) run inside the notebook;
checkpoints/data are written to `Results_task4/` so existing project files are untouched.

In [14]:
import os, math, time, random
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import pandas as pd

import Simulations.config as config
from Simulations.Synthetic_NL_model.parameters import Q_structure, R_structure, m, n, m1x_0, m2x_0, f, h
from Simulations.Extended_sysmdl import SystemModel
from Simulations.utils import DataGen
from Filters.EKF_test import EKFTest
from RobustKalmanPY.robust_kalman import RobustKalman                              # REKF + proposed (use_nn)
from RobustKalmanPY.robust_kalman_original import RobustKalman as RobustKalmanOrig # existing Robust KalmanNet
from Pipelines.Pipeline_EKF import Pipeline_EKF
from KNet.KalmanNet_nn import KalmanNetNN

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
crit = nn.MSELoss()
def to_dB(x): return 10.0 * math.log10(x)

## Configuration & aligned-init policy

In [15]:
args = config.general_settings(); args.use_cuda = False; args.randomLength = False
args.N_E, args.N_CV, args.N_T = 100, 20, 40
args.T, args.T_test = 100, 100
args.n_batch, args.n_steps = 20, 200          # KalmanNet training
args.lr, args.wd = 1e-3, 1e-4
args.in_mult_KNet, args.out_mult_KNet = 5, 40
device = torch.device('cpu')

# proposed Robust KalmanNet training (BPTT); scale up if you want a tighter fit
PROP = dict(epochs=100, n_batch=10, tbptt=50, lr=1e-3, wd=1e-4, gru_hidden=64)

path_results = 'Results_task4/'; os.makedirs(path_results, exist_ok=True)
DataFile = path_results + 'data_task4.pt'     # self-contained; existing project dataset untouched

# noise: nu = q2/r2 = -20 dB, R = 0.1 I (Sec IV-C)
r2 = torch.tensor([0.1]); vv = 10 ** (-20 / 10); q2 = torch.mul(vv, r2)
Q = q2[0] * Q_structure; R = r2[0] * R_structure

# aligned-init policy
X0 = torch.ones(m, 1)                 # aligned filter init = generative x0 (off the x=0 trap)
M2 = 1e-3 * torch.eye(m)             # common initial covariance (matches REKF's V_prev)
CS = [1e-6, 1e-2, 0.05, 0.1, 0.2, 0.5, 1.0]   # Robust EKF c-grid (1e-6 ~ EKF; c=2 diverges)
print(f"N_E={args.N_E} N_T={args.N_T}  KNet epochs={args.n_steps}  proposed epochs={PROP['epochs']}"
      f"  aligned x0=ones, m2x0=1e-3 I")

N_E=100 N_T=40  KNet epochs=200  proposed epochs=100  aligned x0=ones, m2x0=1e-3 I


## System model + shared data

The TRUE model generates the data. The PARTIAL model mismatches `f` only
(`alpha=beta=1, phi=delta=0` -> `sin(x)`); `h` is unchanged; the data is always from
the TRUE model. All filters use the aligned init.

In [16]:
# TRUE (full-information) model
sys_full = SystemModel(f, Q, h, R, args.T, args.T_test, m, n)
sys_full.InitSequence(m1x_0, m2x_0)                 # generative init for DATA generation
sys_full.alpha, sys_full.beta, sys_full.phi, sys_full.delta = 0.9, 1.1, 0.1*math.pi, 0.01
sys_full.a, sys_full.b, sys_full.c = 1, 1, 0

DataGen(args, sys_full, DataFile)
train_input, train_target, cv_input, cv_target, test_input, test_target = torch.load(DataFile, map_location=device)[:6]

# PARTIAL model
def f_partial(x): return torch.sin(x)
sys_part = SystemModel(f_partial, Q, h, R, args.T, args.T_test, m, n)
sys_part.InitSequence(m1x_0, m2x_0)
sys_part.alpha, sys_part.beta, sys_part.phi, sys_part.delta = 1.0, 1.0, 0.0, 0.0
sys_part.a, sys_part.b, sys_part.c = 1, 1, 0

# apply aligned FILTER init + common covariance to both models
for s in (sys_full, sys_part):
    s.m1x_0 = X0.clone(); s.m2x_0 = M2.clone()
print('train', tuple(train_input.shape), ' test', tuple(test_input.shape),
      f" | true f(0)={0.9*math.sin(0.1*math.pi)+0.01:.4f}  partial f(0)=0  h'(0)=0")

train (100, 2, 100)  test (40, 2, 100)  | true f(0)=0.2881  partial f(0)=0  h'(0)=0


## Metric & per-method helpers (posterior MSE -> dB, shared test set)

In [17]:
results = {}
def rec(method, scenario, mdB, extra=''):
    results.setdefault(method, {})[scenario] = mdB
    print(f'  {method:22s} [{scenario:7s}]  MSE = {mdB:+8.3f} dB   {extra}')

def ekf_dB(sysmdl):
    out = EKFTest(args, sysmdl, test_input, test_target)     # out[0] = per-seq posterior MSE
    return to_dB(float(out[0].mean()))

def rekf_sweep(sysmdl):
    row = {}
    for c in CS:
        rk = RobustKalman(sysmdl, test_input[0], float(c), False, False, sl_model=0)  # use_nn=False
        mses = []
        for k in range(test_input.shape[0]):
            rk.c = torch.tensor(float(c)); rk.y = test_input[k]; rk.fnREKF()
            mses.append(crit(rk.Xn_out, test_target[k]).item())
        row[c] = to_dB(float(np.mean(mses)))
    return row, min(row, key=row.get)

def kalmannet_dB(sysmdl, tag):
    torch.manual_seed(SEED)
    KNet = KalmanNetNN(); KNet.NNBuild(sysmdl, args)
    pipe = Pipeline_EKF(tag, 'KNet', 'KNet'); pipe.setssModel(sysmdl); pipe.setModel(KNet)
    pipe.setTrainingParams(args)
    pipe.NNTrain(sysmdl, cv_input, cv_target, train_input, train_target, path_results)  # trained on TRUE data
    out = pipe.NNTest(sysmdl, test_input, test_target, path_results)
    return to_dB(float(out[1]))

def existing_rknet_dB(sysmdl):
    net = RobustKalmanOrig(sysmdl, test_input[0], 1e-3, False, True,
                           input_feat_mode=3, hidden_layers=[20,20,20,20,20], sl_model=0).nn
    mses = []
    for k in range(test_input.shape[0]):
        rk = RobustKalmanOrig(sysmdl, test_input[k], 1e-3, False, True,
                              input_feat_mode=3, hidden_layers=[20,20,20,20,20], sl_model=0)
        rk.nn = net; rk.nn.previous_output = torch.zeros(1, 1)
        with torch.no_grad(): rk.fnREKF(train=False)
        mses.append(crit(rk.Xn, test_target[k]).item())
    return to_dB(float(np.mean(mses)))

def proposed_dB(sysmdl, tag):
    torch.manual_seed(SEED)
    prop = RobustKalman(sysmdl, train_input[0], 1e-3, False, True,
                        input_feat_mode=3, gru_hidden_size=PROP['gru_hidden'], sl_model=0)  # use_nn=True (GRU)
    # exclude the output layer from weight decay so it is not pulled to the sigmoid default
    decay   = [p for nm, p in prop.nn.named_parameters() if 'output_layer' not in nm]
    nodecay = [p for nm, p in prop.nn.named_parameters() if 'output_layer' in nm]
    opt = optim.Adam([{'params': decay, 'weight_decay': PROP['wd']},
                      {'params': nodecay, 'weight_decay': 0.0}], lr=PROP['lr'])
    for ep in range(PROP['epochs']):                              # mini-batch BPTT on posterior state-MSE
        idx = random.sample(range(train_input.shape[0]), min(PROP['n_batch'], train_input.shape[0]))
        opt.zero_grad(); losses = []
        for j in idx:
            prop.y = train_input[j]
            prop.fnREKF(train=True, reset=True, bptt_truncation=PROP['tbptt'])
            losses.append(crit(prop.Xn_out, train_target[j]))
        loss = torch.stack(losses).mean(); loss.backward()
        torch.nn.utils.clip_grad_norm_(prop.nn.parameters(), 1.0); opt.step()
    mses, cbar = [], []
    for k in range(test_input.shape[0]):
        prop.y = test_input[k]
        with torch.no_grad(): prop.fnREKF(train=False, reset=True)
        mses.append(crit(prop.Xn_out, test_target[k]).item())
        cbar.append(float(torch.stack([c.reshape(()) for c in prop.c_array]).mean()))
    return to_dB(float(np.mean(mses))), float(np.mean(cbar))

## 1. EKF + Robust EKF (`c`-sweep) - full + partial

In [18]:
cstar = {}
for scen, sysmdl in [('full', sys_full), ('partial', sys_part)]:
    rec('EKF', scen, ekf_dB(sysmdl))
    row, bc = rekf_sweep(sysmdl); cstar[scen] = bc
    print('      REKF sweep [%s]: ' % scen + '  '.join(f'c{c}:{row[c]:+.2f}' for c in CS)
          + f'   -> c*={bc}')
    rec('Robust EKF', scen, row[bc], f'(c*={bc})')

Extended Kalman Filter - MSE LOSS: tensor(-29.7125) [dB]
Extended Kalman Filter - STD: tensor(0.4557) [dB]
Inference Time: 0.4116506576538086
  EKF                    [full   ]  MSE =  -29.712 dB   
      REKF sweep [full]: c1e-06:-29.29  c0.01:-29.29  c0.05:-29.29  c0.1:-29.28  c0.2:-29.26  c0.5:-29.18  c1.0:-29.01   -> c*=0.01
  Robust EKF             [full   ]  MSE =  -29.295 dB   (c*=0.01)
Extended Kalman Filter - MSE LOSS: tensor(-8.6797) [dB]
Extended Kalman Filter - STD: tensor(0.1083) [dB]
Inference Time: 0.32658886909484863
  EKF                    [partial]  MSE =   -8.680 dB   
      REKF sweep [partial]: c1e-06:-8.74  c0.01:-9.57  c0.05:-10.66  c0.1:-11.46  c0.2:-12.48  c0.5:-13.88  c1.0:-14.62   -> c*=1.0
  Robust EKF             [partial]  MSE =  -14.622 dB   (c*=1.0)


## 2. KalmanNet - trained at the aligned init (on the TRUE data), full + partial

In [19]:
rec('KalmanNet', 'full',    kalmannet_dB(sys_full, 'task4-knet-full'))
rec('KalmanNet', 'partial', kalmannet_dB(sys_part, 'task4-knet-part'))

Epoch 1/200, MSE Training: 0.0048, MSE Validation: 0.0028
Epoch 2/200, MSE Training: 0.0028, MSE Validation: 0.0019
Epoch 3/200, MSE Training: 0.0020, MSE Validation: 0.0015
Epoch 4/200, MSE Training: 0.0015, MSE Validation: 0.0014
Epoch 5/200, MSE Training: 0.0014, MSE Validation: 0.0015
Epoch 6/200, MSE Training: 0.0015, MSE Validation: 0.0017
Epoch 7/200, MSE Training: 0.0017, MSE Validation: 0.0017
Epoch 8/200, MSE Training: 0.0018, MSE Validation: 0.0017
Epoch 9/200, MSE Training: 0.0018, MSE Validation: 0.0016
Epoch 10/200, MSE Training: 0.0017, MSE Validation: 0.0015
Epoch 11/200, MSE Training: 0.0016, MSE Validation: 0.0014
Epoch 12/200, MSE Training: 0.0015, MSE Validation: 0.0013
Epoch 13/200, MSE Training: 0.0014, MSE Validation: 0.0012
Epoch 14/200, MSE Training: 0.0012, MSE Validation: 0.0012
Epoch 15/200, MSE Training: 0.0013, MSE Validation: 0.0012
Epoch 16/200, MSE Training: 0.0013, MSE Validation: 0.0012
Epoch 17/200, MSE Training: 0.0012, MSE Validation: 0.0012
Epoch 

## 3. Existing Robust KalmanNet (professor's original; untrained by construction)

In [20]:
rec('existing Robust KNet', 'full',    existing_rknet_dB(sys_full))
rec('existing Robust KNet', 'partial', existing_rknet_dB(sys_part))

  existing Robust KNet   [full   ]  MSE =  -29.168 dB   
  existing Robust KNet   [partial]  MSE =  -13.908 dB   


## 4. Proposed Robust KalmanNet (GRU-learned `c_t`, mini-batch BPTT)

Trained with the same posterior state-MSE loss on the TRUE data. The v3 head is
`sigmoid -> c in (0,1)`: if the beneficial tolerance is `>= 1` (the sweep above found
`c*` near 1), `c_t` will saturate near the ceiling - see `docs/TASK4_DESIGN.md` for the
`c_range`/`leo` alternative if you want to lift that cap.

In [21]:
for scen, sysmdl in [('full', sys_full), ('partial', sys_part)]:
    mdB, cavg = proposed_dB(sysmdl, f'task4-prop-{scen}')
    rec('proposed Robust KNet', scen, mdB, f'(mean learned c = {cavg:.3f})')

  proposed Robust KNet   [full   ]  MSE =  -29.231 dB   (mean learned c = 0.328)
  proposed Robust KNet   [partial]  MSE =  -14.622 dB   (mean learned c = 1.000)


## 5. Results table and the four qualitative orderings

In [22]:
order = ['EKF', 'Robust EKF', 'KalmanNet', 'existing Robust KNet', 'proposed Robust KNet']
df = pd.DataFrame({sc: {mth: results[mth][sc] for mth in order} for sc in ['full', 'partial']})
print(df.to_string(float_format=lambda x: f'{x:+.3f}'))

def side(a, b, sc):
    va, vb = results[a][sc], results[b][sc]
    return f'{a} ({va:+.2f}) {"<" if va < vb else ">"} {b} ({vb:+.2f})   [{"OK" if (va<vb) else "no"}]'
print('\n--- expected orderings (lower dB = better) ---')
print('FULL    EKF <= RobustEKF          :', side('EKF', 'Robust EKF', 'full'))
print('FULL    KalmanNet <= proposedRKNet:', side('KalmanNet', 'proposed Robust KNet', 'full'))
print('PARTIAL RobustEKF <  EKF          :', side('Robust EKF', 'EKF', 'partial'))
print('PARTIAL proposedRKNet <  KalmanNet:', side('proposed Robust KNet', 'KalmanNet', 'partial'))
df

                        full  partial
EKF                  -29.712   -8.680
Robust EKF           -29.295  -14.622
KalmanNet            -29.698  -21.499
existing Robust KNet -29.168  -13.908
proposed Robust KNet -29.231  -14.622

--- expected orderings (lower dB = better) ---
FULL    EKF <= RobustEKF          : EKF (-29.71) < Robust EKF (-29.29)   [OK]
FULL    KalmanNet <= proposedRKNet: KalmanNet (-29.70) < proposed Robust KNet (-29.23)   [OK]
PARTIAL RobustEKF <  EKF          : Robust EKF (-14.62) < EKF (-8.68)   [OK]
PARTIAL proposedRKNet <  KalmanNet: proposed Robust KNet (-14.62) > KalmanNet (-21.50)   [no]


,full,partial
EKF,-29.712491,-8.679746
Robust EKF,-29.294633,-14.621572
KalmanNet,-29.697553,-21.499234
existing Robust KNet,-29.168116,-13.907578
proposed Robust KNet,-29.231335,-14.621572


## Reading the results

This notebook is the complete Task-4 pipeline: five estimators, two information
regimes, one shared test set, all trained/evaluated inside the notebook. The table
and the ordering checks above report exactly what the code produced - interpret and
decide next steps from these numbers.

Reference expectations from `docs/TASK4_DESIGN.md`: on matched (full) data robustness
is a mild penalty, so `EKF >= RobustEKF` and `KalmanNet >= RobustKalmanNet`; under the
`f`-mismatch, covariance robustness should help the EKF family (`RobustEKF > EKF`). The
proposed method's partial-info standing versus KalmanNet, and its learned `c`, are read
directly from the table above. To scale precision: raise `args.n_steps` (KalmanNet) and
`PROP['epochs']` (proposed); to explore the noise axis, sweep `r2` as the paper does.